# PaveX Auto-Annotation Pipeline

Automatic pothole annotation with **GroundingDINO + Segment Anything (SAM)**, exported as **YOLOv8-seg** labels, plus evaluation on a 200-image ground-truth subset. See the repository README and the paper for details.

**How to run**

| Where | Setup |
|---|---|
| **Google Colab** (free GPU works) | *Runtime → Change runtime type → T4 GPU*. Edit the Colab paths in Cell 0, then *Run all*. |
| **Local** (VS Code / Jupyter) | `pip install -r requirements.txt`, copy `configs/paths.example.json` → `configs/paths.local.json`, edit it, then run all. Without an NVIDIA GPU, use `MAX_IMAGES = 5` for a quick test. |

Each run writes to its own folder `runs/run_YYYYMMDD_HHMM/` with a copy of the config and `run_info.json`, so results are never overwritten.

## 0 · Environment, paths and run settings
Detects Colab vs. local, sets folders, and controls what runs (`RUN_ANNOTATE`, `RUN_EVAL`, `MAX_IMAGES`, `SKIP_EXISTING`).

In [ ]:
# ============================================================
# Cell 0 — Environment, paths and run settings
# Works on Google Colab AND locally (VS Code / Jupyter).
# ============================================================
import sys, json, subprocess
from datetime import datetime
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Colab already ships torch/torchvision with CUDA, so install everything else only.
    # Versions match requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==4.57.1", "accelerate==1.11.0", "timm==1.0.22",
                    "huggingface_hub==0.36.0", "segment-anything==1.0",
                    "opencv-python-headless==4.10.0.84", "tqdm>=4.66.0"], check=True)
# Locally: run `pip install -r requirements.txt` once in your virtual environment instead.

# ---------------- PATHS ----------------
if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    PACK_DIR   = DRIVE_ROOT / "pavex_bbox_experiments" / "exports" / "pavex_sam_seed_pack_v11b"
    PATHS = {
        "config_dir":      PACK_DIR,                                 # pipeline_config.json + seed_input.json
        "sam_weights":     PACK_DIR / "models" / "sam_vit_h_4b8939.pth",
        "images_dir":      DRIVE_ROOT / "BigDataset" / "augmented",  # images to auto-annotate
        "eval_images_dir": DRIVE_ROOT / "BigDataset" / "originals",  # folder with the ORIGINAL images (img_XXXX.jpg)
        "gt_labels_dir":   DRIVE_ROOT / "BigDataset" / "labels_gt",  # 200 ground-truth labels
        "runs_dir":        DRIVE_ROOT / "BigDataset" / "runs",       # every run gets its own subfolder here
    }
else:
    # Local: paths come from configs/paths.local.json (not committed to Git).
    # Copy configs/paths.example.json -> configs/paths.local.json and edit it.
    REPO_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    paths_file = REPO_ROOT / "configs" / "paths.local.json"
    if not paths_file.exists():
        raise FileNotFoundError(
            f"{paths_file} not found. Copy configs/paths.example.json to "
            f"configs/paths.local.json and set your folders."
        )
    raw = json.loads(paths_file.read_text(encoding="utf-8-sig"))  # utf-8-sig tolerates Notepad's BOM
    # Relative paths are resolved from the repo root
    PATHS = {k: (Path(v) if Path(v).is_absolute() else (REPO_ROOT / v).resolve())
             for k, v in raw.items() if not k.startswith("_")}

CONFIG_DIR      = Path(PATHS["config_dir"])
IMG_DIR         = Path(PATHS["images_dir"])
EVAL_IMG_DIR    = Path(PATHS["eval_images_dir"])
GT_LABELS_DIR   = Path(PATHS["gt_labels_dir"])
RUNS_DIR        = Path(PATHS["runs_dir"])

# ---------------- RUN SETTINGS ----------------
# A new, timestamped folder per run means results are never overwritten.
# To RESUME an interrupted run, set RUN_NAME to that run's folder name.
RUN_NAME      = "run_" + datetime.now().strftime("%Y%m%d_%H%M")
RUN_ANNOTATE  = True    # Cell 5: auto-annotate every image in IMG_DIR
RUN_EVAL      = True    # Cell 6: evaluate on the ground-truth subset
MAX_IMAGES    = None    # e.g. 5 for a quick smoke test; None = all images
SKIP_EXISTING = True    # resume: skip images whose label file already exists

RUN_ROOT        = RUNS_DIR / RUN_NAME
PRED_SEG_DIR    = RUN_ROOT / "labels_pred_seg"
OVERLAY_ALL_DIR = RUN_ROOT / "overlays_all_seg"
OVERLAY_GT_DIR  = RUN_ROOT / "overlays_seg_eval"
EVAL_DIR        = RUN_ROOT / "eval"
for d in [PRED_SEG_DIR, OVERLAY_ALL_DIR, OVERLAY_GT_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Environment    :", "Google Colab" if IN_COLAB else "Local")
for k, v in PATHS.items():
    print(f"{k:16s}: {v}  {'' if Path(v).exists() else '<-- NOT FOUND'}")
print("RUN_ROOT        :", RUN_ROOT)

## 1 · Configuration (paper §3.5, Table 2)
Loads thresholds, prompts and post-filters from `pipeline_config.json`; `seed_input.json` overrides the area and solidity filters.

In [ ]:
# ============================================================
# Cell 1 — Read pipeline_config.json + seed_input.json
# ============================================================
import json, shutil, zipfile

# On Colab the pack may still be zipped next to its folder
PACK_ZIP = CONFIG_DIR.parent / f"{CONFIG_DIR.name}.zip"
if PACK_ZIP.exists() and not CONFIG_DIR.exists():
    print("Unzipping", PACK_ZIP.name, "...")
    with zipfile.ZipFile(PACK_ZIP) as zf:
        zf.extractall(CONFIG_DIR.parent)

PIPELINE_CFG = CONFIG_DIR / "pipeline_config.json"
SEED_INPUT   = CONFIG_DIR / "seed_input.json"

if not PIPELINE_CFG.exists():
    raise FileNotFoundError(f"pipeline_config.json not found at {PIPELINE_CFG}")

with open(PIPELINE_CFG, "r") as f:
    pipe = json.load(f)

# ---- MODELS ----
paths_info = pipe.get("paths", {})
HF_REPO    = paths_info.get("hf_repo_grounding_dino", "IDEA-Research/grounding-dino-base")
SAM_ARCH   = pipe.get("models", {}).get("SAM", {}).get("arch", None)

# ---- PROMPTS ----
# Note: the prompts already end in ".", so joining with " . " gives "potholes. . pothole. . ..."
# This is how the paper's results were produced, so it is intentionally left unchanged.
PROMPTS_MP = pipe.get("prompts", ["pothole", "road hole", "damaged road"])

# ---- THRESHOLDS ----
thr = pipe.get("thresholds", {})
TEXT_THR = float(thr.get("TEXT_THR", 0.25))   # kept for reference; does not affect box filtering
BOX_THR  = float(thr.get("BOX_THR",  0.35))
NMS_THR  = float(thr.get("NMS_THR",  0.45))

# ---- POST-FILTERS (mask area, solidity, edge rule) ----
post_filters = pipe.get("post_filters", {})
MAX_AREA_FRAC          = float(post_filters.get("max_area_frac", 0.65))
edge_cfg               = post_filters.get("edge_touches_rule", {})
EDGE_RULE_ON           = bool(edge_cfg.get("on", False))
EDGE_AREA_FRAC_GATE    = float(edge_cfg.get("area_frac_gate", 0.40))
EDGE_MIN_EDGES_TOUCHED = int(edge_cfg.get("min_edges_touched", 3))

# seed_input.json overrides min_area, max_area_frac, solidity_min
MIN_MASK_AREA = 200.0
SOLIDITY_MIN  = 0.48
if SEED_INPUT.exists():
    with open(SEED_INPUT, "r") as f:
        seed_in = json.load(f)
    pf_seed = (seed_in.get("POST_FILTERS") or {})
    MIN_MASK_AREA = float(pf_seed.get("min_area", MIN_MASK_AREA))
    MAX_AREA_FRAC = float(pf_seed.get("max_area_frac", MAX_AREA_FRAC))
    SOLIDITY_MIN  = float(pf_seed.get("solidity_min", SOLIDITY_MIN))

# ---- MASK CONSOLIDATION ----
MASK_NMS_IOU          = 0.70   # drop duplicate masks overlapping more than this
NESTED_CONTAINMENT    = 0.90   # child counts as "inside" if >= 90% of it lies in the parent
NESTED_AREA_RATIO_MAX = 0.70   # ... and the child is at most 70% of the parent's area

# ---- SAM WEIGHTS (fail fast) ----
SAM_WEIGHTS_PATH = Path(PATHS["sam_weights"])
if SAM_ARCH is None:
    raise ValueError("models.SAM.arch missing in pipeline_config.json")
if not SAM_WEIGHTS_PATH.exists():
    raise FileNotFoundError(
        f"SAM weights not found at {SAM_WEIGHTS_PATH}. Download sam_vit_h_4b8939.pth from "
        "https://github.com/facebookresearch/segment-anything and update the path."
    )

# ---- Save a copy of the config inside the run folder (provenance) ----
for p in [PIPELINE_CFG, SEED_INPUT]:
    if p.exists():
        shutil.copy2(p, RUN_ROOT / p.name)

print("HF_REPO          :", HF_REPO)
print("PROMPTS_MP       :", PROMPTS_MP)
print("BOX_THR, NMS_THR :", BOX_THR, NMS_THR)
print("SAM_ARCH         :", SAM_ARCH)
print("SAM_WEIGHTS_PATH :", SAM_WEIGHTS_PATH)
print("--- Post-filters ---")
print("MIN_MASK_AREA    :", MIN_MASK_AREA)
print("MAX_AREA_FRAC    :", MAX_AREA_FRAC)
print("SOLIDITY_MIN     :", SOLIDITY_MIN)
print("EDGE_RULE_ON     :", EDGE_RULE_ON, "| gate:", EDGE_AREA_FRAC_GATE, "| edges:", EDGE_MIN_EDGES_TOUCHED)

## 2 · Load models
GroundingDINO (Hugging Face) for text-prompted boxes and SAM ViT-H for box-prompted masks. The environment is recorded in `run_info.json`.

In [ ]:
# ============================================================
# Cell 2 — Load GroundingDINO & SAM
# ============================================================
import platform
import torch
import transformers
from transformers import AutoProcessor, GroundingDinoForObjectDetection
from segment_anything import sam_model_registry, SamPredictor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device, f"({torch.cuda.get_device_name(0)})" if device == "cuda" else
      "(no NVIDIA GPU: expect roughly 30-60 s per image)")

# GroundingDINO (downloaded from Hugging Face on first run, then cached)
processor   = AutoProcessor.from_pretrained(HF_REPO)
gdino_model = GroundingDinoForObjectDetection.from_pretrained(HF_REPO).to(device).eval()

# SAM
print("Loading SAM:", SAM_ARCH, "from", SAM_WEIGHTS_PATH)
sam_model = sam_model_registry[SAM_ARCH](checkpoint=str(SAM_WEIGHTS_PATH))
sam_model.to(device)
sam_predictor = SamPredictor(sam_model)
print("Models loaded.")

# ---- Record the environment with the run (provenance) ----
def _git_commit():
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                              text=True, check=True).stdout.strip()
    except Exception:
        return None

RUN_INFO = {
    "run_name": RUN_NAME,
    "started": datetime.now().isoformat(timespec="seconds"),
    "environment": "colab" if IN_COLAB else "local",
    "device": device,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "git_commit": _git_commit(),
    "images_dir": str(IMG_DIR),
    "eval_images_dir": str(EVAL_IMG_DIR),
    "max_images": MAX_IMAGES,
}
_ = (RUN_ROOT / "run_info.json").write_text(json.dumps(RUN_INFO, indent=2))  # "_ =" hides the character count

## 3 · Utilities
Box IoU/NMS, mask IoU/Dice, mask NMS and nested-mask suppression (§3.5.4), overlays, and the YOLOv8-seg ground-truth reader.

In [ ]:
# ============================================================
# Cell 3 — Generic utilities: IoU, overlays, masks, GT reader
# ============================================================
import cv2
import numpy as np
from tqdm import tqdm

IMG_EXTS = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

# ------------ BOX IOU / NMS ------------
def iou_xyxy(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h
    if inter == 0:
        return 0.0
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    if union <= 0:
        return 0.0
    return inter / union

def nms_boxes(boxes, scores, iou_thr=0.45):
    if len(boxes) == 0:
        return boxes, scores
    boxes = boxes.copy()
    scores = scores.copy()
    order = scores.argsort()[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        if len(order) == 1:
            break
        rest = order[1:]
        ious = np.array([iou_xyxy(boxes[i], boxes[j]) for j in rest])
        rest = rest[ious < iou_thr]
        order = rest
    keep = np.array(keep, dtype=int)
    return boxes[keep], scores[keep]

# ------------ LABEL PATH → IMAGE PATH (Roboflow-style) ------------
def label_to_image_path(label_path: Path, img_dir: Path):
    """
    Map Roboflow-style labels to image filenames:
      img_0002_jpg.rf.xxxxx.txt  -> <img_dir>/img_0002.jpg (or .png/.jpeg)
    """
    stem = label_path.stem
    img_stem = stem
    for marker in ["_jpg.rf.", "_png.rf.", "_jpeg.rf."]:
        if marker in stem:
            img_stem = stem.split(marker)[0]
            break

    candidates = [img_dir / f"{img_stem}{ext}" for ext in IMG_EXTS]
    for p in candidates:
        if p.exists():
            return p

    print(f"[WARN] No image named {img_stem}.jpg/.png in {img_dir} (for label {label_path.name})")
    return None

# ------------ OVERLAYS ------------
def save_overlay_boxes_and_masks(
    img_path: Path,
    boxes_xyxy,
    masks,
    out_dir: Path,
):
    """
    Draw predicted masks (transparent red) + boxes (green).
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    img = cv2.imread(str(img_path))
    if img is None:
        return
    overlay = img.copy()

    # masks
    if masks is not None:
        for m in masks:
            m_bin = (m > 0).astype(np.uint8)
            if m_bin.sum() == 0:
                continue
            color = np.zeros_like(img)
            color[:, :, 2] = 255  # red
            mask_3 = m_bin[..., None]
            overlay = np.where(
                mask_3 == 1,
                (0.2 * color + 0.8 * overlay).astype(overlay.dtype),
                overlay,
            )

    # boxes
    if boxes_xyxy is not None and len(boxes_xyxy) > 0:
        for (x1, y1, x2, y2) in boxes_xyxy.astype(int):
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)

    out_path = out_dir / img_path.name
    cv2.imwrite(str(out_path), overlay)

# ------------ MASK METRICS ------------
def mask_iou(mask1: np.ndarray, mask2: np.ndarray):
    m1 = (mask1 > 0)
    m2 = (mask2 > 0)
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    if union == 0:
        return 0.0
    return inter / union

def mask_dice(mask1: np.ndarray, mask2: np.ndarray):
    m1 = (mask1 > 0)
    m2 = (mask2 > 0)
    inter = np.logical_and(m1, m2).sum()
    s1 = m1.sum()
    s2 = m2.sum()
    denom = s1 + s2
    if denom == 0:
        return 0.0
    return 2.0 * inter / denom

# ------------ MASK NMS ------------
# Both functions return the INDICES of the masks to keep, so the caller can keep
# masks, boxes and scores aligned (fix: previously only masks were filtered).
def mask_nms_indices(masks, scores, iou_thr=0.7):
    """Greedy mask-level NMS. Returns indices of kept masks, highest score first."""
    if len(masks) == 0:
        return []
    scores = np.asarray(scores, dtype=float)
    order = list(scores.argsort()[::-1])
    keep = []
    while order:
        i = order.pop(0)
        keep.append(int(i))
        order = [j for j in order if mask_iou(masks[i], masks[j]) <= iou_thr]
    return keep


def nested_mask_indices(masks, containment_thr=0.9, area_ratio_thr=0.7):
    """
    Drop 'child' masks that lie mostly inside a larger 'mother' mask.

    A child is dropped when:
      - containment = |child AND mother| / |child| >= containment_thr, and
      - area(child) / area(mother) <= area_ratio_thr.

    Fix: the previous version tested IoU >= 0.7 together with area ratio <= 0.7.
    Because IoU <= area ratio for any pair, both could only hold at exactly 0.7,
    so the rule almost never fired. Containment measures "inside" correctly.
    """
    n = len(masks)
    if n <= 1:
        return list(range(n))
    bins  = [(m > 0) for m in masks]
    areas = np.array([b.sum() for b in bins], dtype=float)
    keep  = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j or not keep[i] or not keep[j]:
                continue
            if areas[i] > areas[j] or (areas[i] == areas[j] and i < j):
                larger, smaller = i, j
            else:
                larger, smaller = j, i
            if areas[smaller] / max(areas[larger], 1e-6) > area_ratio_thr:
                continue
            inter = np.logical_and(bins[smaller], bins[larger]).sum()
            if inter / max(areas[smaller], 1e-6) >= containment_thr:
                keep[smaller] = False
    return [int(k) for k in np.flatnonzero(keep)]

# ------------ YOLOv8-seg GT reader ------------
def read_yolo_seg_masks(label_path: Path, img_w: int, img_h: int):
    if not label_path.exists():
        return []
    text = label_path.read_text().strip()
    if not text:
        return []

    h, w = img_h, img_w
    masks = []

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 7:          # class id + at least 3 points
            continue
        coords = [float(v) for v in parts[1:]]
        if len(coords) % 2 != 0:
            continue
        xs = coords[0::2]
        ys = coords[1::2]
        pts = []
        for x_norm, y_norm in zip(xs, ys):
            x = int(x_norm * w)
            y = int(y_norm * h)
            pts.append([x, y])
        pts = np.array(pts, dtype=np.int32).reshape(-1, 1, 2)
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [pts], 255)
        masks.append(mask)

    return masks

## 4 · Detection and segmentation pipeline (§3.5.1 – §3.5.4)
`predict_image()` runs the full pipeline for one image: GroundingDINO → road-band suppression → box NMS → SAM → mask NMS → nested suppression → post-filters. Cells 5 and 6 both call it, so annotation and evaluation always use identical logic.

In [ ]:
# ============================================================
# Cell 4 — GDINO predictor, SAM masks, mask filters
# ============================================================
def gdino_predict_for_image(
    img_bgr: np.ndarray,
    box_thr: float = BOX_THR,
    text_thr: float = TEXT_THR,
    nms_thr: float = NMS_THR,
    margin_frac: float = 0.02,
    min_road_area_frac: float = 0.20,
    min_road_height_frac: float = 0.30,
    min_road_width_frac: float = 0.30,
):
    """
    Run GroundingDINO on one image (already loaded, BGR) and return filtered boxes.

    - Uses prompts from PROMPTS_MP.
    - Applies a simple road-band filter to remove giant bands across edges.
    """
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    img_area = float(w * h)

    prompt_str = " . ".join(PROMPTS_MP)
    inputs = processor(
        images=img_rgb,
        text=prompt_str,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = gdino_model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs,
        threshold=float(box_thr),
        target_sizes=[(h, w)]
    )[0]

    boxes  = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()

    if len(boxes) == 0:
        return boxes, scores, (w, h)

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    widths  = np.clip(x2 - x1, 0, None)
    heights = np.clip(y2 - y1, 0, None)
    areas   = widths * heights

    area_fracs   = areas / img_area
    height_fracs = heights / float(h)
    width_fracs  = widths  / float(w)

    # valid content edges (ignore small grey border)
    margin_x = int(margin_frac * w)
    margin_y = int(margin_frac * h)

    left_edge_valid   = margin_x
    right_edge_valid  = w - margin_x
    top_edge_valid    = margin_y
    bottom_edge_valid = h - margin_y

    touches_left   = x1 <= left_edge_valid
    touches_right  = x2 >= right_edge_valid
    touches_top    = y1 <= top_edge_valid
    touches_bottom = y2 >= bottom_edge_valid

    horiz_road = (
        touches_left & touches_right &
        (height_fracs >= min_road_height_frac) &
        (area_fracs   >= min_road_area_frac)
    )
    vert_road = (
        touches_top & touches_bottom &
        (width_fracs  >= min_road_width_frac) &
        (area_fracs   >= min_road_area_frac)
    )
    huge_box = area_fracs >= 0.75

    road_like = horiz_road | vert_road | huge_box

    keep_mask = ~road_like
    boxes  = boxes[keep_mask]
    scores = scores[keep_mask]

    if len(boxes) == 0:
        return boxes, scores, (w, h)

    boxes_nms, scores_nms = nms_boxes(boxes, scores, iou_thr=nms_thr)
    return boxes_nms, scores_nms, (w, h)


def sam_masks_from_boxes(img_bgr, boxes_xyxy, scores=None):
    """
    Run SAM for each GDINO box and return (masks, boxes, scores), index-aligned.

    Fix: if SAM returned an empty mask, the old version skipped the mask but kept
    the box, so later masks were paired with the wrong boxes.
    """
    empty = ([], np.zeros((0, 4), dtype=float), np.array([], dtype=float))
    if boxes_xyxy is None or len(boxes_xyxy) == 0:
        return empty

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    sam_predictor.set_image(img_rgb)

    if scores is None:
        scores = np.ones(len(boxes_xyxy), dtype=float)

    masks_out, boxes_out, scores_out = [], [], []
    for box, score in zip(boxes_xyxy, scores):
        box_arr = np.array(box, dtype=np.float32)[None, :]
        with torch.no_grad():
            mask, _, _ = sam_predictor.predict(
                box=box_arr,
                point_coords=None,
                point_labels=None,
                multimask_output=False
            )
        m = mask[0]
        if m is None or m.size == 0:
            continue
        m_bin = (m > 0).astype(np.uint8)
        if m_bin.sum() == 0:
            continue
        masks_out.append(m_bin)
        boxes_out.append(np.asarray(box, dtype=float))
        scores_out.append(float(score))

    if not masks_out:
        return empty
    return masks_out, np.vstack(boxes_out), np.array(scores_out, dtype=float)


def mask_solidity(mask: np.ndarray) -> float:
    """
    Approximate solidity = area / convex_hull_area.
    """
    m = (mask > 0).astype(np.uint8)
    if m.sum() < 5:
        return 0.0
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 0.0
    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)
    if area <= 0:
        return 0.0
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    if hull_area <= 0:
        return 0.0
    return float(area / hull_area)


def filter_masks_with_postfilters(masks, boxes_xyxy, scores,
                                  min_area: float = MIN_MASK_AREA,
                                  max_area_frac: float = MAX_AREA_FRAC,
                                  solidity_min: float = SOLIDITY_MIN,
                                  edge_rule_on: bool = EDGE_RULE_ON,
                                  edge_area_frac_gate: float = EDGE_AREA_FRAC_GATE,
                                  edge_min_edges_touched: int = EDGE_MIN_EDGES_TOUCHED):
    """
    Apply area / solidity / edge-touch filters using config from pack.
    Returns filtered (masks, boxes, scores).
    """
    if len(masks) == 0:
        return [], np.zeros((0, 4), dtype=float), np.array([], dtype=float)

    h, w = masks[0].shape[:2]
    img_area = float(w * h)

    kept_masks = []
    kept_boxes = []
    kept_scores = []

    for m, box, s in zip(masks, boxes_xyxy, scores):
        area = (m > 0).sum()
        if area < min_area:
            continue

        area_frac = area / img_area
        if area_frac > max_area_frac:
            continue

        sol = mask_solidity(m)
        if sol < solidity_min:
            continue

        # optional edge-touch rule (road-like giant region)
        if edge_rule_on:
            x1, y1, x2, y2 = box
            touches_top = y1 <= 0.01 * h
            touches_bottom = y2 >= 0.99 * h
            touches_left = x1 <= 0.01 * w
            touches_right = x2 >= 0.99 * w
            edges_touched = sum([touches_top, touches_bottom, touches_left, touches_right])
            if area_frac >= edge_area_frac_gate and edges_touched >= edge_min_edges_touched:
                # likely huge border region
                continue

        kept_masks.append(m)
        kept_boxes.append(box)
        kept_scores.append(s)

    if not kept_masks:
        return [], np.zeros((0, 4), dtype=float), np.array([], dtype=float)

    return kept_masks, np.vstack(kept_boxes), np.array(kept_scores, dtype=float)


def predict_image(img_bgr):
    """
    Full pipeline for ONE image, used by both auto-annotation (Cell 5) and
    evaluation (Cell 6) so they can never drift apart.

    Returns (masks, boxes, scores); element k of each belongs to the same detection.
    """
    # 1) GroundingDINO boxes + road-band suppression + box NMS
    boxes, scores, _ = gdino_predict_for_image(img_bgr)

    # 2) SAM box-prompted masks (aligned with boxes)
    masks, boxes, scores = sam_masks_from_boxes(img_bgr, boxes, scores)

    # 3) Mask NMS, then nested-mask suppression; index-based so boxes stay aligned
    keep = mask_nms_indices(masks, scores, iou_thr=MASK_NMS_IOU)
    masks, boxes, scores = [masks[k] for k in keep], boxes[keep], scores[keep]

    keep = nested_mask_indices(masks, containment_thr=NESTED_CONTAINMENT,
                               area_ratio_thr=NESTED_AREA_RATIO_MAX)
    masks, boxes, scores = [masks[k] for k in keep], boxes[keep], scores[keep]

    # 4) Post-filters (area, solidity, edge rule)
    return filter_masks_with_postfilters(masks, boxes, scores)

## 5 · Auto-annotate all images (§3.5.5)
Writes one YOLOv8-seg label (`cls x1 y1 … xn yn`) per image and a QA overlay. Interrupted runs can be resumed by re-running with the same `RUN_NAME`.

In [ ]:
# ============================================================
# Cell 5 — Binary mask → YOLOv8-seg label + auto-annotate all
# ============================================================
import os

def binary_mask_to_yolov8_seg_line(
    mask,
    img_w: int,
    img_h: int,
    cls_id: int = 0,
    simplify_eps: float = 1.5,
):
    """
    Convert a binary mask (H,W) into one YOLOv8-seg label line:
      cls x1 y1 x2 y2 ... xn yn      (normalised to [0, 1])

    Fix: the old version also wrote "cx cy w h" after the class id. YOLOv8-seg
    reads every number after the class id as polygon points, so those four values
    became two fake vertices and corrupted every mask.
    """
    m = (mask > 0).astype(np.uint8)
    if m.sum() == 0:
        return None

    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    cnt = max(contours, key=cv2.contourArea)       # largest connected region only
    if cv2.contourArea(cnt) < 10:
        return None

    if simplify_eps > 0:
        approx = cv2.approxPolyDP(cnt, simplify_eps, True)
        if len(approx) >= 3:
            cnt = approx

    pts = cnt.reshape(-1, 2).astype(float)
    if len(pts) < 3:
        return None

    xs = np.clip(pts[:, 0] / img_w, 0.0, 1.0)
    ys = np.clip(pts[:, 1] / img_h, 0.0, 1.0)

    parts = [str(cls_id)]
    for x, y in zip(xs, ys):
        parts += [f"{x:.6f}", f"{y:.6f}"]
    return " ".join(parts)


def _write_atomic(path: Path, text: str):
    """Write to a temp file, then rename. An interrupted run never leaves a half-written label."""
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text)
    os.replace(tmp, path)


def list_images(folder: Path):
    # Deduplicate: Windows file systems are case-insensitive, so *.jpg and *.JPG
    # would otherwise return the same files twice.
    found = {p.resolve() for ext in IMG_EXTS for p in folder.glob(f"*{ext}")}
    return sorted(found)


def auto_annotate_all_images():
    img_paths = list_images(IMG_DIR)
    if MAX_IMAGES:
        img_paths = img_paths[:MAX_IMAGES]
    print(f"Found {len(img_paths)} images in {IMG_DIR}")

    n_done = n_skipped = n_failed = 0
    for img_path in tqdm(img_paths, desc="Auto-annotating"):
        out_txt = PRED_SEG_DIR / f"{img_path.stem}.txt"
        if SKIP_EXISTING and out_txt.exists():
            n_skipped += 1
            continue

        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            print(f"[WARN] Failed to read {img_path}")
            n_failed += 1
            continue
        h, w = img_bgr.shape[:2]

        masks, boxes, scores = predict_image(img_bgr)

        save_overlay_boxes_and_masks(img_path, boxes, masks, out_dir=OVERLAY_ALL_DIR)

        lines = [binary_mask_to_yolov8_seg_line(m, img_w=w, img_h=h, cls_id=0, simplify_eps=1.5)
                 for m in masks]
        lines = [l for l in lines if l is not None]
        # An empty file means "image processed, no pothole found"
        _write_atomic(out_txt, "\n".join(lines) + "\n" if lines else "")
        n_done += 1

    # Completeness check: every image should now have a label file
    n_labels = len(list(PRED_SEG_DIR.glob("*.txt")))
    print(f"Done. Annotated: {n_done} | skipped (already done): {n_skipped} | unreadable: {n_failed}")
    print(f"Label files: {n_labels} / {len(img_paths)} images",
          "" if n_labels >= len(img_paths) - n_failed else "<-- INCOMPLETE, re-run this cell to resume")
    print("Labels in  :", PRED_SEG_DIR)
    print("Overlays in:", OVERLAY_ALL_DIR)


if RUN_ANNOTATE:
    auto_annotate_all_images()
else:
    print("RUN_ANNOTATE is False; skipping auto-annotation.")

## 6 · Evaluation on the ground-truth subset (§3.7)
Greedy one-to-one matching at mask IoU ≥ 0.5; precision/recall are micro-averaged, IoU/Dice are averaged over matched instances only. Ground-truth labels belong to the **original** images, so `eval_images_dir` must contain them.

In [ ]:
# ============================================================
# Cell 6 — Segmentation-based evaluation on 200 GT images
# ============================================================
import csv
import json as _json_mod

IOU_THR_EVAL = 0.5

def eval_dataset_on_gt_seg():
    # Never overwrite earlier results (see README: results are tied to one run)
    if (EVAL_DIR / "seg_eval_summary.json").exists():
        raise FileExistsError(f"Results already exist in {EVAL_DIR}. Use a new RUN_NAME.")

    gt_label_paths = sorted(GT_LABELS_DIR.glob("*.txt"))
    if MAX_IMAGES:
        gt_label_paths = gt_label_paths[:MAX_IMAGES]
    print(f"Found {len(gt_label_paths)} GT label files")

    per_img_stats = []
    skipped_labels = []

    total_TP = total_FP = total_FN = 0
    neg_imgs = 0
    total_FP_neg = 0
    ious_all = []
    dices_all = []

    for lbl_path in tqdm(gt_label_paths, desc="Evaluating SEG on GT subset"):
        # GT labels belong to ORIGINAL images, so look in EVAL_IMG_DIR (not the augmented folder)
        img_path = label_to_image_path(lbl_path, EVAL_IMG_DIR)
        if img_path is None:
            skipped_labels.append(lbl_path.name)
            continue

        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            print(f"[WARN] Cannot read image {img_path}")
            skipped_labels.append(lbl_path.name)
            continue
        h, w = img_bgr.shape[:2]

        # Predictions: the same pipeline as Cell 5
        masks_pred, boxes_pred, mask_scores = predict_image(img_bgr)
        img_w, img_h = w, h
        n_pred = len(masks_pred)

        # GT masks
        masks_gt = read_yolo_seg_masks(lbl_path, img_w, img_h)
        n_gt = len(masks_gt)

        # Overlays for QA on GT subset
        save_overlay_boxes_and_masks(img_path, boxes_pred, masks_pred, out_dir=OVERLAY_GT_DIR)

        # Negative GT image
        if n_gt == 0:
            neg_imgs += 1
            total_FP_neg += n_pred
            total_FP += n_pred
            per_img_stats.append({
                "image": img_path.name,
                "n_gt": n_gt,
                "n_pred": n_pred,
                "TP": 0,
                "FP": n_pred,
                "FN": 0,
                "IoU_mean": float("nan"),
                "Dice_mean": float("nan"),
            })
            continue

        # GT has masks but predictions none
        if n_pred == 0:
            total_FN += n_gt
            per_img_stats.append({
                "image": img_path.name,
                "n_gt": n_gt,
                "n_pred": n_pred,
                "TP": 0,
                "FP": 0,
                "FN": n_gt,
                "IoU_mean": float("nan"),
                "Dice_mean": float("nan"),
            })
            continue

        # IoU & Dice matrix
        iou_mat = np.zeros((n_gt, n_pred), dtype=float)
        dice_mat = np.zeros((n_gt, n_pred), dtype=float)
        for i in range(n_gt):
            for j in range(n_pred):
                iou_val = mask_iou(masks_gt[i], masks_pred[j])
                dice_val = mask_dice(masks_gt[i], masks_pred[j])
                iou_mat[i, j] = iou_val
                dice_mat[i, j] = dice_val

        matched_gt = set()
        matched_pred = set()
        tps = 0
        ious_img = []
        dices_img = []

        pairs = []
        for i in range(n_gt):
            for j in range(n_pred):
                pairs.append((iou_mat[i, j], i, j))
        pairs.sort(reverse=True, key=lambda x: x[0])

        for iou_val, i, j in pairs:
            if iou_val < IOU_THR_EVAL:
                break
            if i in matched_gt or j in matched_pred:
                continue
            matched_gt.add(i)
            matched_pred.add(j)
            tps += 1
            ious_img.append(iou_val)
            dices_img.append(dice_mat[i, j])

        fps = n_pred - tps
        fns = n_gt - tps

        total_TP += tps
        total_FP += fps
        total_FN += fns

        if ious_img:
            ious_all.extend(ious_img)
            dices_all.extend(dices_img)
            iou_mean_img = float(np.mean(ious_img))
            dice_mean_img = float(np.mean(dices_img))
        else:
            iou_mean_img = float("nan")
            dice_mean_img = float("nan")

        per_img_stats.append({
            "image": img_path.name,
            "n_gt": n_gt,
            "n_pred": n_pred,
            "TP": tps,
            "FP": fps,
            "FN": fns,
            "IoU_mean": iou_mean_img,
            "Dice_mean": dice_mean_img,
        })

    # Fail fast: if no image could be evaluated, stop instead of saving an all-zero "result"
    n_evaluated = len(gt_label_paths) - len(skipped_labels)
    if n_evaluated == 0:
        raise FileNotFoundError(
            f"None of the {len(gt_label_paths)} GT labels had a matching image in {EVAL_IMG_DIR}. "
            f"Check that eval_images_dir points to the folder with the ORIGINAL images "
            f"(e.g. img_0002.jpg). Nothing was saved."
        )

    precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0.0
    recall    = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0.0
    iou_mean_dataset  = float(np.mean(ious_all))  if ious_all else float("nan")
    dice_mean_dataset = float(np.mean(dices_all)) if dices_all else float("nan")
    fp_per_neg_img = total_FP_neg / neg_imgs if neg_imgs > 0 else float("nan")

    summary = {
        "TP": total_TP,
        "FP": total_FP,
        "FN": total_FN,
        "precision": precision,
        "recall": recall,
        "IoU_mean": iou_mean_dataset,
        "Dice_mean": dice_mean_dataset,
        "neg_imgs": neg_imgs,
        "FP_per_neg_img": fp_per_neg_img,
        "skipped_labels": skipped_labels,
    }

    # Save to disk (for comparison across runs)
    # Per-image CSV with Python's built-in csv module (no pandas needed)
    fieldnames = []
    for row in per_img_stats:
        fieldnames += [k for k in row if k not in fieldnames]
    with open(EVAL_DIR / "seg_eval_per_image.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(per_img_stats)
    # NaN is not valid JSON, so write it as null
    summary_json = {k: (None if isinstance(v, float) and np.isnan(v) else v) for k, v in summary.items()}
    with open(EVAL_DIR / "seg_eval_summary.json", "w") as f:
        _json_mod.dump(summary_json, f, indent=2)

    if skipped_labels:
        print(f"[WARN] {len(skipped_labels)} GT labels had no matching image in {EVAL_IMG_DIR}")
    print("Saved per-image stats  ->", EVAL_DIR / "seg_eval_per_image.csv")
    print("Saved summary JSON     ->", EVAL_DIR / "seg_eval_summary.json")
    return per_img_stats, summary


if RUN_EVAL:
    per_img_stats_seg, summary_seg = eval_dataset_on_gt_seg()
    try:
        from IPython.display import display
        display(summary_seg)
    except ImportError:
        print(summary_seg)
else:
    print("RUN_EVAL is False; skipping evaluation.")